# AutomatedScheduling — ML Development Notebook

End-to-end walkthrough for building and validating the ML layer.

**Purpose:** Develop and validate all models using synthetic data so the pipeline is production-ready when real data is plugged in. Real data only requires swapping the data-loading cells.

**Models covered:**
1. **Demand Forecasting** (XGBoost) — predict transactions, sales, required headcount per 30-min slot
2. **Preference Score** (XGBoost Regressor) — learn each barista's latent shift preferences from schedule history
3. **No-Show Risk** (XGBoost Classifier) — predict probability an employee doesn't show up for a specific assignment

**Additional features beyond the README (explored in Section 5):**
- No-show / call-out risk scoring
- Employee fatigue modeling (rolling consecutive days worked)
- Productivity differentials (transactions per labor-hour by employee)
- Team pairing dynamics
- New hire / probationary employee rules
- Pay rate optimization signals
- Implicit preference learning from manager overrides

## Section 1 — Import Required Libraries

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))   # repo root on path

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, roc_auc_score)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

print("Libraries loaded ✓")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")
print(f"  xgboost {xgb.__version__}")

## Section 2 — Synthetic Data Generation (Placeholder for Real Data)

The generators below produce data in the **exact schema** your real CSVs must follow.
When real data is ready, replace the `generate_all()` call with a loader that reads from your database or CSV exports — the rest of the notebook stays unchanged.

**Real data schema requirements:**

| File | Key columns |
|------|-------------|
| `demand_history.csv` | `location_id, date, datetime, hour, minute, day_of_week, month, is_holiday, is_promotion, is_rain, transactions, sales` |
| `employees.csv` | `employee_id, location_id, role, skills, employment_type, desired_weekly_hours, preference, avail_start_hour, avail_end_hour, unavailable_days, productivity_score, no_show_risk, fatigue_sensitivity, pay_rate, tenure_months, is_probationary` |
| `schedule_history.csv` | `location_id, employee_id, date, day_of_week, shift_type, shift_start_hour, shift_end_hour, scheduled_hours, no_show, manager_override, week_start, weekly_hours_accumulated` |

In [ ]:
from ml.data.data_generator import generate_all

# Generate 2 years of synthetic data and cache to disk
DATA_DIR = "../data"
datasets = generate_all(output_dir=DATA_DIR)

demand_df    = datasets["demand"]
employees_df = datasets["employees"]
schedules_df = datasets["schedules"]

print(f"\nDemand rows   : {len(demand_df):,}")
print(f"Employees     : {len(employees_df)}")
print(f"Schedule rows : {len(schedules_df):,}")
demand_df.head(3)

In [ ]:
# Quick EDA — daily transaction totals
demand_df["date"] = pd.to_datetime(demand_df["date"])
daily = demand_df.groupby("date")["transactions"].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(daily["date"], daily["transactions"], linewidth=0.8, alpha=0.8)
axes[0].set_title("Daily Total Transactions (2 years)")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Transactions")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()

dow_avg = demand_df.groupby(["day_of_week", "hour"])["transactions"].mean().reset_index()
pivot = dow_avg.pivot(index="hour", columns="day_of_week", values="transactions")
pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
sns.heatmap(pivot, ax=axes[1], cmap="YlOrRd", annot=False)
axes[1].set_title("Avg Transactions — Hour × Day of Week")
axes[1].set_ylabel("Hour of Day")

plt.tight_layout()
plt.show()

## Section 3 — Feature Engineering: Sales History

Features extracted from raw demand data:
- **Temporal**: hour, minute, day-of-week, month, week-of-year, quarter, is_weekend, is_monday, is_friday
- **Cyclical encodings**: sin/cos transforms prevent the model treating hour 23 as far from hour 0
- **Lag features**: same-slot values from 1 week, 2 weeks, and 4 weeks ago — the strongest predictors
- **Rolling stats**: 4-week rolling mean and std for the same slot type
- **Contextual**: holiday flag, promotion flag, rain flag
- **Derived target**: `required_headcount` = ⌈transactions / 18⌉ + 2 (1 supervisor + 1 register partner)

In [ ]:
from ml.features.feature_engineering import DemandFeatureBuilder

demand_builder = DemandFeatureBuilder()
demand_featured = demand_builder.build(demand_df)

feature_cols = demand_builder.get_feature_columns()
target_cols  = demand_builder.get_target_columns()

# Drop rows where lag features are NaN (first ~4 weeks)
demand_featured = demand_featured.dropna(subset=feature_cols).reset_index(drop=True)

print(f"Rows after dropping lag NaNs: {len(demand_featured):,}")
print(f"\nFeature columns ({len(feature_cols)}):")
for i, c in enumerate(feature_cols):
    print(f"  {i+1:2d}. {c}")

demand_featured[feature_cols + target_cols].describe().round(2)

## Section 4 — Feature Engineering: Barista Preferred Hours & Schedule History

The preference model learns **latent preferences** — what an employee actually works happily, inferred from:
- Which shift types they've historically been assigned most often (rolling 8-week window)
- Whether they showed up (`no_show = 0`) — no-shows signal discomfort
- Whether a manager overrode their assignment (`manager_override = 1`) — signals a mismatch
- Their declared preference label (morning / midday / evening / flexible)

**Target variable — `preference_score`:**
| Value | Meaning |
|-------|---------|
| `1.0` | Employee accepted shift, showed up, no manager override |
| `0.5` | Employee had a no-show on this shift type (uncertain preference) |
| `0.0` | Manager overrode the assignment (implicit preference mismatch) |

In [ ]:
from ml.features.feature_engineering import PreferenceFeatureBuilder

pref_builder = PreferenceFeatureBuilder()
pref_featured = pref_builder.build(schedules_df, employees_df)

pref_feature_cols = pref_builder.get_feature_columns()
pref_target_col   = pref_builder.get_target_column()

pref_featured = pref_featured.dropna(
    subset=pref_feature_cols + [pref_target_col, "no_show"]
).reset_index(drop=True)

print(f"Preference feature rows: {len(pref_featured):,}")
print(f"\nTarget distribution (preference_score):")
print(pref_featured[pref_target_col].value_counts().sort_index())

# Visualise: how preference scores vary by shift type
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pref_by_shift = pref_featured.groupby("shift_type")[pref_target_col].mean().sort_values()
pref_by_shift.plot(kind="barh", ax=axes[0])
axes[0].set_title("Mean Preference Score by Shift Type")
axes[0].set_xlabel("Mean Score")

pref_featured.groupby("preference")["preference_score"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="steelblue"
)
axes[1].set_title("Mean Preference Score by Declared Preference")
axes[1].set_xlabel("Mean Score")
plt.tight_layout()
plt.show()

## Section 5 — Additional Features Not Yet in the README

Below are **8 high-value features** identified beyond what the README currently specifies, along with how they are encoded in this pipeline.

In [ ]:
"""
Additional features demonstrated below:

  1. No-show / call-out risk score        → no_show_risk column (employee attribute)
  2. Employee fatigue: rolling consecutive days worked
  3. Productivity differential            → productivity_score column
  4. New hire / probationary flag         → is_probationary column
  5. Pay rate (cost optimization signal)  → pay_rate column
  6. Manager override signal              → manager_override (implicit dislike label)
  7. Cyclical time encodings              → already in demand features (sin/cos)
  8. Consecutive days fatigue proxy       → computed below
"""

# --- Feature 2: Consecutive days worked (fatigue proxy) ---
schedules_df["date"] = pd.to_datetime(schedules_df["date"])

def max_consecutive_days(dates):
    if len(dates) == 0:
        return 0
    dates = sorted(dates.dt.date.unique())
    max_run = run = 1
    for i in range(1, len(dates)):
        run = run + 1 if (dates[i] - dates[i-1]).days == 1 else 1
        max_run = max(max_run, run)
    return max_run

fatigue_df = (
    schedules_df.groupby("employee_id")
    .apply(lambda g: pd.Series({
        "max_consecutive_days": max_consecutive_days(g["date"]),
        "total_shifts": len(g),
        "no_show_rate": g["no_show"].mean(),
        "override_rate": g["manager_override"].mean(),
    }))
    .reset_index()
)

# Merge with employee info
fatigue_merged = fatigue_df.merge(
    employees_df[["employee_id", "preference", "employment_type",
                  "productivity_score", "pay_rate", "is_probationary"]],
    on="employee_id"
)

print("Employee fatigue & risk profile (sample):")
print(fatigue_merged.sort_values("max_consecutive_days", ascending=False).head(10).to_string(index=False))

# --- Visualise no-show rate vs productivity ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(fatigue_merged["productivity_score"],
                fatigue_merged["no_show_rate"], alpha=0.6)
axes[0].set_title("Productivity vs No-Show Rate")
axes[0].set_xlabel("Productivity Score")
axes[0].set_ylabel("No-Show Rate")

axes[1].hist(fatigue_merged["max_consecutive_days"], bins=15, edgecolor="white")
axes[1].set_title("Max Consecutive Days Worked Distribution")
axes[1].set_xlabel("Days")
axes[1].set_ylabel("Employees")

fatigue_merged.groupby("preference")["no_show_rate"].mean().sort_values().plot(
    kind="barh", ax=axes[2]
)
axes[2].set_title("No-Show Rate by Declared Preference")
axes[2].set_xlabel("Avg No-Show Rate")

plt.tight_layout()
plt.show()

## Section 6 — Data Preprocessing & Train/Test Split

In [ ]:
# ── Demand model split (temporal — last 8 weeks as hold-out) ─────────────
X_demand = demand_featured[feature_cols]
y_demand  = demand_featured[target_cols]

HOLD_OUT = 8 * 7 * 34   # 8 weeks × 7 days × 34 thirty-min slots
X_d_train, X_d_test = X_demand.iloc[:-HOLD_OUT], X_demand.iloc[-HOLD_OUT:]
y_d_train, y_d_test = y_demand.iloc[:-HOLD_OUT], y_demand.iloc[-HOLD_OUT:]

print(f"Demand  → Train: {len(X_d_train):,}  Test: {len(X_d_test):,}")

# ── Preference model split (random — not time-sensitive) ─────────────────
X_pref   = pref_featured[pref_feature_cols]
y_pref   = pref_featured[pref_target_col]
y_noshow = pref_featured["no_show"].astype(int)

X_p_train, X_p_test, yp_train, yp_test, yn_train, yn_test = train_test_split(
    X_pref, y_pref, y_noshow,
    test_size=0.20, random_state=42
)

print(f"Pref    → Train: {len(X_p_train):,}  Test: {len(X_p_test):,}")
print(f"No-show rate in train: {yn_train.mean():.3f}  test: {yn_test.mean():.3f}")

# Verify no NaNs remain
assert X_d_train.isna().sum().sum() == 0, "NaNs found in demand train set!"
assert X_p_train.isna().sum().sum() == 0, "NaNs found in preference train set!"
print("\nNo NaN values in feature matrices ✓")

## Section 7 — Model Architecture

### Model 1: Demand Forecasting — XGBoost Multi-Output Regressor
- One `XGBRegressor` per target (`transactions`, `sales`, `required_headcount`)
- Shared feature matrix, separate trees
- Early stopping on 10% validation tail

### Model 2: Preference Score — XGBoost Regressor
- Predicts `preference_score ∈ [0, 1]`
- Trained on implicit feedback from schedule history
- Output used as soft-constraint weight in the OR-Tools optimizer

### Model 3: No-Show Risk — XGBoost Classifier
- Predicts `P(no_show = 1)`
- Class-imbalance handled via `scale_pos_weight`
- Output used to hedge coverage on high-risk assignments

## Section 8 — Train the Models

In [ ]:
from ml.models.demand_forecasting_model import DemandForecastingModel
from ml.models.preference_model import PreferenceScoreModel, NoShowRiskModel

# ── Model 1: Demand Forecasting ───────────────────────────────────────────
print("Training Demand Forecasting Model ...")
demand_model = DemandForecastingModel()
demand_model.fit(X_d_train, y_d_train)
print("  Done ✓")

# ── Model 2: Preference Score ─────────────────────────────────────────────
print("Training Preference Score Model ...")
pref_model = PreferenceScoreModel()
pref_model.fit(X_p_train, yp_train)
print("  Done ✓")

# ── Model 3: No-Show Risk ─────────────────────────────────────────────────
print("Training No-Show Risk Model ...")
noshow_model = NoShowRiskModel()
noshow_model.fit(X_p_train, yn_train)
print("  Done ✓")

## Section 9 — Evaluate Model Performance

In [ ]:
# ── Demand model metrics ──────────────────────────────────────────────────
demand_metrics = demand_model.evaluate(X_d_test, y_d_test)
print("Demand Forecasting Metrics (Hold-Out Test Set):")
print(demand_metrics.to_string())

# Actual vs Predicted — transactions (first 2 days of hold-out)
test_dates = demand_featured["datetime"].iloc[-HOLD_OUT:].values
preds_demand = demand_model.predict(X_d_test)

PLOT_N = 2 * 34
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(test_dates[:PLOT_N], y_d_test["transactions"].values[:PLOT_N],
             label="Actual", linewidth=2)
axes[0].plot(test_dates[:PLOT_N], preds_demand["transactions"].values[:PLOT_N],
             label="Predicted", linewidth=1.5, linestyle="--", alpha=0.9)
axes[0].set_title("Demand Forecast — Transactions (First 2 Hold-Out Days)")
axes[0].set_ylabel("Transactions / 30 min")
axes[0].legend()

axes[1].plot(test_dates[:PLOT_N], y_d_test["sales"].values[:PLOT_N],
             label="Actual", linewidth=2)
axes[1].plot(test_dates[:PLOT_N], preds_demand["sales"].values[:PLOT_N],
             label="Predicted", linewidth=1.5, linestyle="--", alpha=0.9)
axes[1].set_title("Demand Forecast — Sales ($)")
axes[1].set_ylabel("Sales ($)")
axes[1].legend()

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%a %H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────
fi = demand_model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

fi["transactions"].sort_values().tail(15).plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 15 Features — Transactions Model")
axes[0].set_xlabel("Importance")

fi["required_headcount"].sort_values().tail(15).plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Top 15 Features — Headcount Model")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

# ── Predicted headcount heatmap (hour × day of week) ─────────────────────
test_copy = X_d_test.copy()
test_copy["predicted_headcount"] = preds_demand["required_headcount"].values
test_copy["hour"]       = demand_featured["hour"].iloc[-HOLD_OUT:].values
test_copy["day_of_week"] = demand_featured["day_of_week"].iloc[-HOLD_OUT:].values

pivot = test_copy.pivot_table(
    values="predicted_headcount", index="hour", columns="day_of_week", aggfunc="mean"
)
pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
ax.set_title("Predicted Required Headcount — Hour × Day of Week")
ax.set_ylabel("Hour of Day")
plt.tight_layout()
plt.show()

In [ ]:
# ── Preference Score Model ────────────────────────────────────────────────
pref_metrics = pref_model.evaluate(X_p_test, yp_test)
print("Preference Score Model Metrics:", pref_metrics)

pref_preds = pref_model.predict(X_p_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(yp_test.values, pref_preds, alpha=0.25, s=8, edgecolors="none")
axes[0].plot([0, 1], [0, 1], "r--", lw=1.5)
axes[0].set_title("Preference Score: Actual vs Predicted")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")

pref_model.get_feature_importance().head(12).sort_values().plot(
    kind="barh", ax=axes[1], color="steelblue"
)
axes[1].set_title("Top 12 Features — Preference Score Model")
axes[1].set_xlabel("Importance")
plt.tight_layout()
plt.show()

# ── No-Show Risk Model ────────────────────────────────────────────────────
noshow_metrics = noshow_model.evaluate(X_p_test, yn_test)
print("\nNo-Show Risk Model Metrics:", noshow_metrics)

noshow_probs = noshow_model.predict_proba(X_p_test)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(noshow_probs, bins=40, edgecolor="white")
axes[0].axvline(0.5, color="red", linestyle="--", label="Threshold 0.5")
axes[0].set_title("No-Show Risk Score Distribution")
axes[0].set_xlabel("Predicted No-Show Probability")
axes[0].set_ylabel("Count")
axes[0].legend()

noshow_model.get_feature_importance().head(12).sort_values().plot(
    kind="barh", ax=axes[1], color="coral"
)
axes[1].set_title("Top 12 Features — No-Show Risk Model")
axes[1].set_xlabel("Importance")
plt.tight_layout()
plt.show()

## Section 10 — Save & Export Models

All three models are serialized to `models/artifacts/` using `joblib`. The `SchedulingInference` class wraps all three and is the single import the scheduling engine needs.

In [ ]:
ARTIFACTS_DIR = "../models/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

demand_model.save(os.path.join(ARTIFACTS_DIR, "demand_forecasting_model.pkl"))
pref_model.save(os.path.join(ARTIFACTS_DIR, "preference_score_model.pkl"))
noshow_model.save(os.path.join(ARTIFACTS_DIR, "noshow_risk_model.pkl"))

print("\nAll models saved ✓")
print(f"  → {ARTIFACTS_DIR}/demand_forecasting_model.pkl")
print(f"  → {ARTIFACTS_DIR}/preference_score_model.pkl")
print(f"  → {ARTIFACTS_DIR}/noshow_risk_model.pkl")

In [ ]:
from ml.inference import SchedulingInference

# Load all models via the unified inference interface
inference = SchedulingInference.load(ARTIFACTS_DIR)
print("SchedulingInference loaded ✓")

# ── Demo 1: Forecast demand for the next week ─────────────────────────────
# In production: pass a raw DataFrame for the upcoming week's time slots
sample_demand = demand_df.tail(7 * 34).copy()   # 7 days of raw demand rows
forecast = inference.forecast_demand(sample_demand)
print(f"\nForecast sample (first 5 rows):")
print(forecast.head().to_string(index=False))

# ── Demo 2: Score candidate assignments ───────────────────────────────────
# The scheduling engine will generate candidate (employee, shift) pairs
# and pass them here for soft-constraint weighting
sample_assignments = pref_featured[pref_feature_cols + ["employee_id", "shift_type",
                                                          "day_of_week",
                                                          "shift_start_hour",
                                                          "shift_end_hour"]].head(10).copy()
scores = inference.score_assignments(sample_assignments)
print(f"\nAssignment scores (sample):")
print(scores[["employee_id", "shift_type", "preference_score",
              "noshow_risk", "assignment_value"]].to_string(index=False))